# Spatial Autocorrelation Exploration: Moran's I and Local Moran's I

## Purpose

This notebook summarizes Global Moran's I and Local Moran's I / LISA results for tract-level arrest activity in Durham.

This is a spatial statistical analysis, not a classification model. It supports exploratory spatial analysis and decision-support interpretation by evaluating spatial autocorrelation and local spatial association patterns.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "ml").exists():
    repo_root = Path.cwd().parents[1]

summary_path = repo_root / "ml" / "outputs" / "morans_i_global_summary.json"
local_results_path = repo_root / "ml" / "outputs" / "local_morans_i_results.csv"
static_map_path = repo_root / "ml" / "outputs" / "local_morans_i_static_map.png"

with summary_path.open("r", encoding="utf-8") as file:
    summary = json.load(file)

local_results = pd.read_csv(local_results_path)

print("Rows:", len(local_results))
print("Columns:", list(local_results.columns))
print("Analysis variable:", summary["analysis_variable"])

## Global Moran's I Summary

In [ ]:
global_summary = pd.DataFrame(
    [
        {
            "Moran's I": summary.get("morans_i", summary.get("global_morans_i")),
            "Expected I": summary["expected_i"],
            "z_sim": summary.get("z_sim"),
            "p_sim": summary["p_sim"],
            "permutations": summary["permutations"],
            "spatial weights": summary.get("spatial_weights", summary["spatial_weights_method"]),
            "weights transform": summary["weights_transform"],
            "number of tracts": summary["n_tracts"],
        }
    ]
)
global_summary

Moran's I = 0.2443 and `p_sim = 0.001`. This indicates statistically significant positive spatial autocorrelation in tract-level arrest rates.

In practical terms, nearby tracts tend to have more similar arrest-rate values than expected under spatial randomness. This should be interpreted cautiously as an exploratory spatial pattern.

## Local Moran's I / LISA Cluster Summary

In [ ]:
cluster_order = ["Not significant", "High-High", "Low-Low", "High-Low", "Low-High"]
cluster_counts = (
    local_results["lisa_cluster"]
    .value_counts()
    .reindex(cluster_order, fill_value=0)
    .rename_axis("lisa_cluster")
    .reset_index(name="count")
)
cluster_counts

## Significant Local Spatial Associations

In [ ]:
significant_clusters = local_results.loc[
    local_results["lisa_cluster"] != "Not significant",
    [
        "tract_geoid",
        "arrests_per_1000_population",
        "local_moran_i",
        "local_moran_p_sim",
        "lisa_cluster",
    ],
].sort_values("arrests_per_1000_population", ascending=False)

significant_clusters

High-High means a high arrest-rate tract near other high arrest-rate tracts. Low-Low means a low arrest-rate tract near other low arrest-rate tracts. High-Low means a relatively high-rate tract near lower-rate neighboring tracts.

These are local spatial association patterns, not causal explanations.

## Static LISA Cluster Map

In [ ]:
display(Image(filename=str(static_map_path)))

## Relationship to the ML Models

Logistic regression, random forest, and PCA logistic regression classify elevated tract-level arrest activity from contextual indicators.

Moran's I / LISA evaluates whether observed arrest rates are spatially clustered across neighboring tracts. The methods answer different but complementary questions.

## Limitations and Responsible Use

- Only 68 census tracts.
- Results depend on Queen contiguity spatial weights.
- Tract boundaries affect results.
- Arrest data reflects enforcement and administrative activity, not direct harm.
- Local clusters do not explain causes.
- This is not an individual-level risk model and should not be used as an operational enforcement tool.

## Portfolio Talking Point

I extended the dashboard's offline analytics layer with Global Moran's I and Local Moran's I / LISA to evaluate whether observed tract-level arrest rates were spatially autocorrelated. The analysis found statistically significant positive global spatial autocorrelation and identified localized High-High, Low-Low, and High-Low spatial association patterns. I treated the results as exploratory spatial evidence rather than operational predictions.